# Introduction

Let's start with the most obvious **problems** that can be seen by just looking at the database
and here are they:
#### LinkedIn
- [x] the `posted_since` column is relative to the collection date, not absolute
- [x] in the `seniority_level` column there's a value called "Not Applicable"
- [ ] Extract the `salary` **PAIN**
#### UpWork
- [x] There are empty strings in the `skills` column
- [ ] there are non **ASCII** charchters in the `description` column
- [x] A lot of columns have useless string components such as *"1978 <ins>hours worked</ins>"*
- [x] Money is represented using strings
- [x] Sometimes values are null and sometimes are strings indicating empty value.
#### Guru
- [x] the `earnings` and `feedback_percent` columns are represented as strings

# Setting up

In [73]:
import pandas as pd
import matplotlib.pyplot as plt

from datetime import datetime, timedelta, date
from transformers import pipeline
from typing import Optional
import sqlite3
import time
import ast
import re
import os

RAW_DB_URI = "file:./data/raw_database.db?mode=ro"
CLEAN_DB_PATH = "./data/clean_database.db"

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
with sqlite3.connect(RAW_DB_URI, uri=True) as con:
    linkedin_df = pd.read_sql_query("SELECT * FROM linkedin", con)
    upwork_df = pd.read_sql_query("SELECT * FROM upwork", con)
    guru_df = pd.read_sql_query("SELECT * FROM guru", con)

In [3]:
linkedin_df.sample(5)

,id,posting_title,location,posted_since,company_name,description,job_url,company_url,applicants,industries,employment_type,job_function,seniority_level,searched_country,searched_job_title
508,4295908314,Administrative Assistant,United States,3 days ago,TRX,Our client is seeking a highly organized and p...,https://www.linkedin.com/jobs/view/administrat...,https://uk.linkedin.com/company/trx-internatio...,Over 200 applicants,Administrative and Support Services and Utilities,Full-time,Administrative,Not Applicable,United States,Data entry
551,4296981763,Administrative Assistant,"Washington, DC",1 day ago,Robert Half,We are looking for a detail-oriented Administr...,https://www.linkedin.com/jobs/view/administrat...,https://www.linkedin.com/company/robert-half-i...,Be among the first 25 applicants,Staffing and Recruiting,Temporary,Administrative,Entry level,United States,Data entry
712,4296082538,"Business Data Scientist, Machine Learning","Washington, DC",2 days ago,Google,This role may also be located in our Playa Vis...,https://www.linkedin.com/jobs/view/business-da...,https://www.linkedin.com/company/google?trk=pu...,None,"Information Services and Technology, Informati...",Full-time,"General Business, Strategy/Planning, and Consu...",Not Applicable,United States,Data scientist
272,4294330247,Data Scientist,"San Ġiljan, Saint Julian's, Malta",5 days ago,ComeOn Group,"ComeOn Group in short:\nFounded in 2008, ComeO...",https://mt.linkedin.com/jobs/view/data-scienti...,https://mt.linkedin.com/company/comeon-group?t...,Be among the first 25 applicants,IT Services and IT Consulting,Full-time,Analyst,Associate,European Union,Data scientist
685,4293014869,Data Engineer,"Nashville, OR",1 week ago,Embold Health,DATA ENGINEERS - TWO OPENINGS\nLocation: Nashv...,https://www.linkedin.com/jobs/view/data-engine...,https://www.linkedin.com/company/emboldhealth?...,Over 200 applicants,Hospitals and Health Care,Full-time,Information Technology,Entry level,United States,Data engineer


# Data cleaning

### LinkedIn

Fixing the `posted_since` column to use dates instead of days since the data was collected<br>
NOTE: it will still be an approximatation because linkedin doesn't specify actual posting date

In [4]:
linkedin_df["posted_since"].unique()

array(['5 days ago', '7 months ago', '3 weeks ago', '2 days ago',
       '2 weeks ago', '1 week ago', '6 days ago', '4 months ago',
       '3 days ago', '4 days ago', '1 month ago', '3 months ago',
       '4 weeks ago', '2 months ago', '5 months ago', '1 day ago',
       '13 hours ago', '18 hours ago', '22 hours ago', '23 hours ago',
       '21 hours ago', '4 hours ago', '14 hours ago', '12 hours ago',
       '17 hours ago', '1 hour ago', '3 hours ago', '7 hours ago',
       '15 hours ago', '16 hours ago', '5 hours ago', '2 hours ago',
       '8 hours ago', '10 hours ago', '9 hours ago'], dtype=object)

In [5]:
collection_time = datetime.fromtimestamp(os.path.getctime("./data/linkedin_jobs.csv"))

hour_pattern  = re.compile(r"^[0-9]+ hour")
day_pattern  = re.compile(r"^[0-9]+ day")
week_pattern  = re.compile(r"^[0-9]+ week")
month_pattern = re.compile(r"^[0-9]+ month")
year_pattern  = re.compile(r"^[0-9]+ year")

value_pattern = re.compile(r"^[0-9]+")

def parse_posted_since(collection_time: datetime, posted_since: str) -> date:
    used_pattern: re.Pattern = None
    
    for pattern in [hour_pattern, day_pattern, week_pattern,
                    month_pattern, year_pattern]:
        if re.match(pattern, posted_since):
            used_pattern = pattern
            break

    if used_pattern == None:
        raise ValueError("The `posted_since` param has invalid form that can't be parsed.")

    interval_value = int(value_pattern.search(posted_since).group())
    interval_unit: timedelta = datetime.hour
    
    if used_pattern == hour_pattern:
        interval_unit = timedelta(hours=1)

    elif used_pattern == day_pattern:
        interval_unit = timedelta(days=1)
        
    elif used_pattern == week_pattern:
        interval_unit = timedelta(weeks=1)
        
    elif used_pattern == month_pattern:
        interval_unit = timedelta(days=29.53)
        
    elif  used_pattern == year_pattern:
        interval_unit = timedelta(days=365.25)

    else:
        raise ValueError("The `posted_since` param has invalid form that can't be parsed.")

    interval = interval_value * interval_unit
    
    return datetime.date(collection_time - interval)

In [6]:
linkedin_df["posted_since"] = linkedin_df["posted_since"].apply(
    lambda x: parse_posted_since(collection_time, posted_since=x))

In [7]:
linkedin_df["posted_since"].sample(5)

832    2025-09-06
520    2025-09-06
193    2025-09-05
381    2025-09-05
778    2025-09-06
Name: posted_since, dtype: object

Replacing the "Not Applicable" value in the `seniority_level` column with None

In [8]:
linkedin_df["seniority_level"].unique()

array(['Not Applicable', 'Entry level', 'Mid-Senior level', 'Associate',
       'Internship', 'Director', 'Executive'], dtype=object)

In [9]:
linkedin_df["seniority_level"] = linkedin_df["seniority_level"].replace(
    "Not Applicable", None)

Now let's try to extract the salary

In [78]:
?re.findall

Signature: re.findall(pattern, string, flags=0)
Docstring:
Return a list of all non-overlapping matches in the string.

If one or more capturing groups are present in the pattern, return
a list of groups; this will be a list of tuples if the pattern
has more than one group.

Empty matches are included in the result.
File:      /usr/local/lib/python3.12/re/__init__.py
Type:      function

In [74]:
desc = linkedin_df["description"].sample(1).iloc[0]

qa_model = pipeline("question-answering")

# I know the following regex won't hanle extreme edge cases but I don't need that for
# this analysis.
matches = re.findall(
    r"([0-9k,\.]+) *[\$|€]|[\$|€] *([0-9k,\.]+)",
    desc,
    flags=re.IGNORECASE
)

if matches:
    salary_values = []
    for match in matches:
        salary_str = match[0] if match[0] else match[1]
        salary_str = salary_str.replace(",", "")
        salary_str = salary_str.lower()

        magnitude = 1.0
        if "k" in salary_str:
            salary_str = salary_str.replace("k", "")
            magnitude = 1000
            
        salary_values.append(float(salary_str) * magnitude)
else:
    print("Did't find a match")

print(desc.replace("$", "######").replace("€", "#####"))
print(salary_values)

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

NameError: name 'torch' is not defined

### UpWork

Fixing the empty strings in the `skills` column

In [10]:
list(upwork_df["skills"].sample(1))

["['Data Analysis', 'Data Science', 'Machine Learning', 'Python', 'PyTorch', 'JavaScript', 'TensorFlow', '', '', '']"]

In [11]:
upwork_df["skills"] = upwork_df["skills"].apply(
    lambda list_: str(list(filter(lambda s: len(s) > 0, ast.literal_eval(list_))))
)

In [12]:
list(upwork_df["skills"].sample(1))

["['Copy & Paste', 'Data Entry', 'Data Mining', 'Microsoft Excel', 'Transaction Data Entry', 'Data Cleaning']"]

Now let's remove the clutter strings from some of the columns

In [13]:
upwork_df[["hours_worked", "hourly_jobs_done", "fixed_jobs_done"]].sample(5)

,hours_worked,hourly_jobs_done,fixed_jobs_done
408,926 hours worked,52 hourly jobs,256 fixed price jobs
536,No hours worked,No hourly jobs,11 fixed price jobs
287,1991 hours worked,19 hourly jobs,30 fixed price jobs
314,60 hours worked,10 hourly jobs,6 fixed price jobs
406,255 hours worked,10 hourly jobs,104 fixed price jobs


In [14]:
def extract_value(s: str) -> int | None:
    value_pattern = re.compile("[0-9]+")

    if not(isinstance(s, str)):
        return None

    match = value_pattern.search(s)

    if not(match):
        return None

    return int(match.group())

for col in ["hours_worked", "hourly_jobs_done", "fixed_jobs_done"]:
    upwork_df[col] = upwork_df[col].apply(extract_value)

In [15]:
upwork_df["hours_worked"].sample(5)

92     516.0
71       NaN
10       NaN
149     64.0
584    111.0
Name: hours_worked, dtype: float64

Converting the money format from being a string into being a float for the `hour_rate` and `earnings`<br>
columns

In [16]:
upwork_df[["earnings", "hour_rate"]].head(5)

,earnings,hour_rate
0,None,$4.8
1,$2K+ earned,$3.5
2,$10K+ earned,$5
3,$100K+ earned,$5
4,None,$5


In [17]:
hour_rate_pattern = re.compile(r"\$[0-9.]+")
earnings_pattern = re.compile(r"\$[0-9]+")

def extract_earnings(s: str) -> int | None:
    if not(isinstance(s, str)):
        return None

    match = earnings_pattern.search(s)
    if not(match):
        return None

    magnitude = 1
    if "K" in s:
        magnitude = 1000
    elif "M" in s:
        magnitude = 1000_000

    return int(match.group()[1:]) * magnitude

def extract_hour_rate(s: str) -> float | None:
    if not(isinstance(s, str)):
        return None

    match = hour_rate_pattern.search(s)

    if not(match):
        return None

    return float(match.group()[1:])

upwork_df["earnings"] = upwork_df["earnings"].apply(extract_earnings)
upwork_df["hour_rate"] = upwork_df["hour_rate"].apply(extract_hour_rate)

In [18]:
upwork_df[["earnings", "hour_rate"]].head(5)

,earnings,hour_rate
0,NaN,4.8
1,2000.0,3.5
2,10000.0,5.0
3,100000.0,5.0
4,NaN,5.0


### Guru

Let's fix the `feedback_percent` and `earnings` format & dtype

In [19]:
guru_df[["feedback_percent", "earnings"]].head(5)

,feedback_percent,earnings
0,None,$0
1,100%,$28K
2,100%,$65K
3,100%,$489K
4,98.8%,$28K


In [20]:
def extract_feedback(s: str) -> float | None:
    pattern = re.compile(r"[0-9.]+")

    if not(isinstance(s, str)):
        return None

    match = pattern.search(s)

    if not(match):
        return None

    return float(match.group())

def extract_earnings(s: str) -> int | None:
    pattern = re.compile(r"\$[0-9,]+")

    if not(isinstance(s, str)):
        return None

    match = pattern.search(s)

    if not(match):
        return None

    magnitude = 1
    if "K" in s:
        magnitude = 1000
    elif "M" in s:
        magnitude = 1000_000
    
    earnings_str = match.group()[1:].replace(",", ".")

    return float(earnings_str) * magnitude

guru_df["feedback_percent"] = guru_df["feedback_percent"].apply(extract_feedback)
guru_df["earnings"] = guru_df["earnings"].apply(extract_earnings)

In [21]:
guru_df[["feedback_percent", "earnings"]].head(5)

,feedback_percent,earnings
0,NaN,0.0
1,100.0,28000.0
2,100.0,65000.0
3,100.0,489000.0
4,98.8,28000.0


# Data storing

In [ ]:
with sqlite3.connect(CLEAN_DB_PATH) as con:
    linkedin_df.to_sql("linkedin", con, if_exists="fail", index=False)
    upwork_df.to_sql("upwork", con, if_exists="fail", index=False)
    guru_df.to_sql("guru", con, if_exists="fail", index=False)